### Calculating weights

Three-stage design weights (`calculate_three_stage_weights()` in `src/funcs.py`):

1. **Clusters**: stratified PPS. Cluster weight = 1 / (1 − (1 − p<sub>i</sub>)<sup>k<sub>h</sub></sup>), where p<sub>i</sub> is the cluster's `proportion` renormalised within its stratum and k<sub>h</sub> is the number of clusters surveyed in that stratum (no separate stratum weight: `USE_STRATUM_WEIGHT = False`). With `PPS_CERTAINTY_UNITS`, a cluster with k<sub>h</sub>·p<sub>i</sub> ≥ 1 is a certainty unit (weight 1) and the others are k<sub>h</sub> − 1 draws from the rest of the stratum
2. **Households**: convenience selection, approximated as (cluster population / household size by setting) / households interviewed. Dividing by households *interviewed* also absorbs household non-response within each cluster. Under PPS the cluster population cancels between stages 1 and 2, leaving N<sub>h</sub> / (k<sub>h</sub> · household size · households interviewed) -- `check_stage_weights()` verifies this
3. **Respondent**: one adult drawn at random (redrawn until an available member is found), so weight = eligible adults in the household

Then normalisation to mean 1 (`normalized_weight_design`), then post-stratification to the frame population by `POSTSTRATIFY_ON` (province): w · N<sub>h</sub> / N̂<sub>h</sub>, renormalised (`normalized_weight`). (Benin only: an extra draw-attempt adjustment, draw attempts / eligible adults capped at 1, kept so Benin's weights are unchanged.)

Clusters link across files on `CLUSTER_KEY` (the settlement name for Togo, which has no cluster ids), compared without accents, case or punctuation.

**Inputs** (`src/config.py`): `DIR_LOCATED_DATA` (from 02a), `DIR_SETTLEMENTS_ELIGIBLE`, `DIR_SETTLEMENTS_CHOSEN`, `SETTLEMENTS_*_RENAME`, `CLUSTER_KEY`, `HOUSEHOLD_SIZE_BY_SETTING`, `CLUSTERS_ALLOCATED`, `PPS_CERTAINTY_UNITS`, `POSTSTRATIFY_ON`.
**Outputs**: `DIR_WEIGHTED_DATA`; cluster-level weights in `4_validation_checks/02b_cluster_weights.csv` and the stage 1 × 2 check in `4_validation_checks/02b_stage12_check.csv`. Check the result with `02b_validate_weights.ipynb`.

In [ ]:
import pandas as pd
from src.config import *
from src.funcs import *
from src.funcs import check_stage_weights

In [ ]:
## Eligible sampling frame + the clusters respondents were finally assigned to in 02a (DIR_SETTLEMENTS_CHOSEN:
## reviewed `selected_name`, one settlement-level point each). Each cluster is linked to one frame row -- by its
## point first, then by name / village -- which gives its population (the frame's X_sum), stratum and proportion.
## The frame is rebuilt as the sample draw saw it (EXCLUDED_CLUSTERS, STRATA_MAP, MIN_CLUSTER_POPULATION).
## Raises with a list if a cluster can't be linked, or matches several frame settlements.
settlements_eligible, settlements_chosen = load_sampling_frames(
    DIR_SETTLEMENTS_ELIGIBLE, DIR_SETTLEMENTS_CHOSEN,
    eligible_rename=SETTLEMENTS_ELIGIBLE_RENAME, chosen_rename=SETTLEMENTS_CHOSEN_RENAME,
    exclude_clusters=EXCLUDED_CLUSTERS, strata_map=STRATA_MAP, min_population=MIN_CLUSTER_POPULATION,
    key=CLUSTER_KEY, point_col=SETTLEMENTS_CHOSEN_POINT_COL,
)
settlements_chosen.drop(columns='geometry', errors='ignore').to_csv(
    DIR_VALIDATE / "02b_clusters_linked_to_frame.csv", index=False, encoding='utf-8-sig')   # cluster-level only
settlements_chosen.groupby('province').agg(clusters=('cluster_key', 'size'), population=('population', 'sum'))

#### Load survey data and pre-process
- remove duplicate submissions
- exclude respondents whose cluster isn't in the surveyed-clusters file
- create household ids
- get number of eligible adults in HH

In [ ]:
# 02a_correct_locations.ipynb assigned each respondent a cluster (RESPONDENT_CLUSTER_COL) -- compared with the
# sampling files the same way load_sampling_frames does (no accents/case/punctuation). Names that the loader
# merged into another (same frame settlement) follow it.
from src.locations import normalize_place

data = pd.read_csv(DIR_LOCATED_DATA)
data['cluster_key'] = (data[RESPONDENT_CLUSTER_COL].where(data[RESPONDENT_CLUSTER_COL].notna())
                       .astype('string').str.strip().map(normalize_place, na_action='ignore')
                       .replace(settlements_chosen.attrs.get('key_aliases', {})))

n_dup = data['KEY'].duplicated().sum()
print(f"{len(data)} respondents; dropping {n_dup} duplicate submission KEYs")
data = data.drop_duplicates(subset='KEY', keep='first')

# Respondents without a cluster, or in one that isn't in the surveyed-clusters file, can't be weighted.
no_cluster = ~data['cluster_key'].isin(settlements_chosen['cluster_key'])
if no_cluster.any():
    print(f"⚠️  {no_cluster.sum()} respondents without a surveyed cluster -- excluded:")
    print(data.loc[no_cluster].groupby(RESPONDENT_CLUSTER_COL, dropna=False).size())
    data.loc[no_cluster].to_csv(DIR_VALIDATE / "02b_respondents_without_cluster.csv", index=False, encoding='utf-8-sig')
data = data.loc[~no_cluster].copy()

In [ ]:
# Create HH IDs
data = create_household_id(data)

##### Explore eligible household members

In [ ]:
# Number of eligible adults listed in the household (last index in hh_member_index_list, minus 1)
data['eligible_adults_in_hh'] = data.hh_member_index_list.map(parse_n).fillna(1)

n_invalid = (data['eligible_adults_in_hh'] < 1).sum()
if n_invalid:
    print(f"⚠️  {n_invalid} households with <1 eligible adult -- check hh_member_index_list format. Setting to 1.")
    data.loc[data['eligible_adults_in_hh'] < 1, 'eligible_adults_in_hh'] = 1

print("Eligible adults per household:")
print(data['eligible_adults_in_hh'].value_counts().sort_index())
print(f"Mean {data['eligible_adults_in_hh'].mean():.2f}, median {data['eligible_adults_in_hh'].median():.0f}")

#### Compute survey weights

In [ ]:
# Stage-2 household size: by cluster from its respondents' majority urban/rural setting
# (HOUSEHOLD_SIZE_BY_SETTING), else AVG_HOUSEHOLD_SIZE everywhere
household_size = AVG_HOUSEHOLD_SIZE
if HOUSEHOLD_SIZE_BY_SETTING and URBAN_RURAL_COL not in data.columns:
    print(f"⚠️  No '{URBAN_RURAL_COL}' column -- using AVG_HOUSEHOLD_SIZE ({AVG_HOUSEHOLD_SIZE}) for every cluster")
elif HOUSEHOLD_SIZE_BY_SETTING:
    setting = data[URBAN_RURAL_COL].astype('string').str.lower().map(
        lambda v: 'urban' if 'urb' in v else 'rural' if 'rur' in v else pd.NA, na_action='ignore')
    cluster_setting = setting.groupby(data['cluster_key']).agg(lambda s: s.mode().iat[0] if s.notna().any() else pd.NA)
    household_size = cluster_setting.map(HOUSEHOLD_SIZE_BY_SETTING).astype(float).fillna(AVG_HOUSEHOLD_SIZE)
    print(f"Clusters by setting: {cluster_setting.value_counts(dropna=False).to_dict()} -> household size "
          f"{HOUSEHOLD_SIZE_BY_SETTING} (no setting: {AVG_HOUSEHOLD_SIZE})")

data_weighted, cluster_weights = calculate_three_stage_weights(
    settlements_eligible, settlements_chosen, data,
    avg_household_size=household_size, inclusion_probability='proportion',
    na_province_path=DIR_VALIDATE / "02b_respondents_without_stratum.csv", cluster_key='cluster_key',
    use_stratum_weight=USE_STRATUM_WEIGHT, certainty_units=PPS_CERTAINTY_UNITS,
)

# Stage 1 x stage 2 should collapse to N_h / (k_h * hh_size * households visited); prints by province only
_ = check_stage_weights(settlements_eligible, settlements_chosen, data_weighted,
                        target_clusters=CLUSTERS_ALLOCATED, out_path=DIR_VALIDATE / "02b_stage12_check.csv")

In [ ]:
# Post-stratification (Statistics Canada 7.1.4.1): w * N_c / N^_c per cell, with N_c the frame population, so
# weighted shares match it. Same formula for PPS as for SRS. The design-only weight stays in normalized_weight_design.
# Factors far from 1 point at problems upstream (see the check above) rather than something to absorb silently.
data_weighted['normalized_weight_design'] = data_weighted['normalized_weight']
if POSTSTRATIFY_ON:
    controls = settlements_eligible.groupby(POSTSTRATIFY_ON)['population'].sum()
    data_weighted['normalized_weight'] = poststratify(data_weighted, controls, cell_col=POSTSTRATIFY_ON,
                                                      weight_col='normalized_weight_design')
# The pipeline (pipeline_output config WEIGHT_COL) weights by combined_weight_adjusted, so apply the
# same per-cell factor there too -- otherwise post-stratification never reaches the dashboard figures.
data_weighted['combined_weight_design'] = data_weighted['combined_weight_adjusted']
if POSTSTRATIFY_ON:
    data_weighted['combined_weight_adjusted'] = (data_weighted['combined_weight_design']
        * data_weighted['normalized_weight'] / data_weighted['normalized_weight_design'])

#### Checks

In [ ]:
not_surveyed = settlements_chosen.loc[~settlements_chosen['cluster_key'].isin(data_weighted['cluster_key']), 'Name']
print(f"Surveyed clusters with no respondents: {sorted(not_surveyed)}")

n_nan = data_weighted['normalized_weight'].isna().sum()
print(f"Respondents without a weight: {n_nan} / {len(data_weighted)}")

# Unweighted vs weighted share by stratum: design weight, then final (post-stratified) weight
pop = settlements_eligible.groupby('province')['population'].sum()
(data_weighted.groupby('province')
    .agg(respondents=('KEY', 'size'), design=('normalized_weight_design', 'sum'), final=('normalized_weight', 'sum'))
    .assign(population_share=pop / pop.sum(),
            sample_share=lambda d: d.respondents / d.respondents.sum(),
            design_share=lambda d: d.design / d.design.sum(),
            final_share=lambda d: d.final / d.final.sum())
    .drop(columns=['design', 'final'])
    .round(3))

##### Export

In [ ]:
data_weighted.to_csv(DIR_WEIGHTED_DATA, index=False)
cluster_weights.to_csv(DIR_VALIDATE / "02b_cluster_weights.csv", index=False)   # cluster-level, no respondent data
print(f"Wrote {len(data_weighted)} respondents to {DIR_WEIGHTED_DATA}")